# TODO

In [89]:
## TODO: add visibility example

## TODO:check if there is an image for sample and count the available pixel

## TODO: count by category to evaluate and correctly interprete. for example at 20 samples

## derive object speed

## TODO: may be rename ann_df?

## TODO: for complex code writhe the conceptual algo in plain text


# Objective

The task considered in this project is **3D object detection on the nuScenes dataset**, with a particular focus on the role of **sensor fusion**. For each scene, the objective is to detect relevant traffic participants and estimate their **class, 3D position, size, and orientation**.

Because nuScenes provides multiple sensing modalities, the problem is not only about achieving accurate object detection, but also about understanding how **LiDAR, cameras, and radar** contribute differently to perception. **LiDAR** provides strong geometric and localization cues, **cameras** provide semantic and appearance information, and **radar** can contribute complementary motion-related and range information.

The exploratory data analysis is therefore designed to study detection difficulty across **object classes, distance, visibility level, and scene conditions**, and to identify the regimes in which fusion is likely to provide meaningful benefit over unimodal baselines.

**Goal:** place the **correct 3D bounding box** around each relevant object in the environment surrounding the ego vehicle.

# Setup and imports

In [90]:
# Setup and imports
import os
import sys
import logging
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple, Literal  # required for Python 3.8

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from nuscenes.nuscenes import NuScenes
from IPython.display import display
from tqdm.auto import tqdm

In [91]:
# Notebook display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [92]:
# Environment check
print(f"Python version: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Python version: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
CUDA available: True
CUDA device count: 1
GPU: NVIDIA GeForce RTX 4090


# Data loading

In [93]:
# Dataset configuration
RUNNING_ON: Literal["local", "hpc"] = "hpcz"
DATASET: Literal["mini", "full"] = "full"

In [94]:
# Resolve dataset path and version
def get_nuscenes_config(running_on: str, dataset: str) -> Tuple[str, str]:
    if running_on == "local":
        if dataset == "mini":
            return "../data/sets/nuscenes", "v1.0-mini"
        if dataset == "full":
            return "../data/sets/nuscenes_full", "v1.0-trainval"

    if running_on == "hpc":
        if dataset == "mini":
            return "/storage/homefs/ae04q066/datasets/nuscenes_mini", "v1.0-mini"
        if dataset == "full":
            return "/rs_scratch/users/ae04q066/nuscenes_full", "v1.0-trainval"

    raise ValueError(
        "Invalid combination: RUNNING_ON must be 'local' or 'hpc', "
        "and DATASET must be 'mini' or 'full'"
    )

DATAROOT, VERSION = get_nuscenes_config(RUNNING_ON, DATASET)

if not os.path.isdir(DATAROOT):
    raise FileNotFoundError(f"DATAROOT not found: {os.path.abspath(DATAROOT)}")

print(f"RUNNING_ON: {RUNNING_ON}")
print(f"DATASET:    {DATASET}")
print(f"DATAROOT:   {DATAROOT}")
print(f"VERSION:    {VERSION}")

ValueError: Invalid combination: RUNNING_ON must be 'local' or 'hpc', and DATASET must be 'mini' or 'full'

In [ ]:
# Load nuScenes
nusc: NuScenes = NuScenes(
    version=VERSION,
    dataroot=str(DATAROOT),
    verbose=True,
) 

# Quick dataset summary
print(f"Loaded version: {nusc.version}")
print(f"Number of logs: {len(nusc.log)}")
print(f"Number of scenes: {len(nusc.scene)}")
print(f"Available tables: {nusc.table_names}")

# Dataset overview

This section introduces the structure of the nuScenes dataset and the main tables used in the project. Since nuScenes is organized as a relational database, understanding the links between tables is important before starting preprocessing and feature engineering.

The analysis focuses especially on the tables related to **samples**, **sensor data**, and **object annotations**, since these are the core components of the 3D detection task.

Relevant links:
* https://www.nuscenes.org/nuscenes
* https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/schema_nuscenes.md
* https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb

## Database schema

nuScenes is organized as a set of linked tables describing scenes, sensor measurements, object instances, and annotations.

The most relevant tables for this project are:

1. `scene` - a roughly 20-second driving segment.
2. `sample` - an annotated snapshot of a scene at a particular timestamp.
3. `sample_data` - sensor data recorded for one sensor at one sample.
4. `sample_annotation` - one annotated object in a given sample.
5. `instance` - a physical object tracked over time.
6. `category` - the semantic class of an object.
7. `attribute` - time-varying properties of an object.
8. `visibility` - visibility level of an annotation.
9. `ego_pose` - ego vehicle pose at a timestamp.
10. `sensor` and `calibrated_sensor` - sensor definitions and calibration.
11. `log` and `map` - recording context and map information.

The schema below is adapted from the official nuScenes tutorial and documentation.

**Source:** Official nuScenes tutorial and schema documentation.

![](https://www.nuscenes.org/public/images/nuscenes-schema.svg)

## Load nuScenes tables into pandas

To make exploration easier, the nuScenes tables are loaded into a dictionary of pandas DataFrames. This provides a tabular view of the relational structure and simplifies later preprocessing steps.

In [ ]:
# Load nuScenes tables into a dictionary of DataFrames
nusc_tables: Dict[str, pd.DataFrame] = {
    table_name: pd.DataFrame(getattr(nusc, table_name))
    for table_name in nusc.table_names
}

table_summary_df: pd.DataFrame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
        }
        for table_name, df in nusc_tables.items()
    ]
).sort_values("table_name").reset_index(drop=True)

table_summary_df

In [ ]:
# High-level dataset inspection
scene_df: pd.DataFrame = nusc_tables["scene"]
category_df: pd.DataFrame = nusc_tables["category"]
attribute_df: pd.DataFrame = nusc_tables["attribute"]

print(f"Number of scenes: {len(scene_df)}")
display(scene_df[["name", "description"]].head())

print(f"\nNumber of categories: {len(category_df)}")
display(category_df[["name", "description"]].head())

print(f"\nNumber of attributes: {len(attribute_df)}")
display(attribute_df[["name", "description"]].head())

## Table relationships relevant for 3D detection

The following relationships are especially important for this project:

1. `scene` -> many `sample`
2. `sample` -> one annotated timestamp
3. `sample_data` -> one sensor measurement for one sample
4. `sample_annotation` -> one object annotation for one sample

In other words, a sample represents a single moment in time, `sample_data` stores the sensor-specific observations at that moment, and `sample_annotation` stores the annotated objects associated with that sample.

In [ ]:
# Inspect one sample and its associated sensor data tokens
sample_df: pd.DataFrame = nusc_tables["sample"]

sample_data_tokens: Dict[str, str] = sample_df.loc[0, "data"]
pprint(sample_data_tokens)

sample_data_token: Optional[str] = sample_data_tokens.get("CAM_BACK")
print("CAM_BACK token:", sample_data_token)

In [ ]:
# Inspect the sample_data table
sample_data_df: pd.DataFrame = nusc_tables["sample_data"]
sample_data_df.head()

## Annotation table and detection targets

For the 3D detection task, the central table is `sample_annotation`. Each row corresponds to one annotated object in one sample.

The detection target is given by the object category, stored in the `category_name` field. In addition to the class label, this table contains the information needed for 3D detection, including object location, size, orientation, and support from LiDAR and radar points.

Key fields in `sample_annotation` include:

- `sample_token`: links the annotation to a sample
- `instance_token`: links repeated annotations of the same object over time
- `category_name`: semantic class of the object
- `visibility_token`: visibility level
- `translation`: 3D box center
- `size`: box dimensions
- `rotation`: box orientation
- `num_lidar_pts`: number of LiDAR points inside the box
- `num_radar_pts`: number of radar points inside the box
- `prev` / `next`: temporal links to the same object in adjacent frames

In [ ]:
# Load and rename the annotation table

sample_annotation_df: pd.DataFrame = nusc_tables["sample_annotation"].rename(
    columns={
        "token": "annotation_token",
        "prev": "prev_ann",
        "next": "next_ann",
    }
)

sample_annotation_df.head()

## Detection classes

The object classes to detect are given by the `category_name` field in the annotation table.

In [ ]:
# Inspect the detection classes

class_names: List[str] = sorted(sample_annotation_df["category_name"].unique().tolist())

print("Number of classes:", len(class_names))
print("\nList of classes:")
class_names

# Data preprocessing

This section prepares the annotation table for later analysis by adding cleaned and derived variables. In particular, it enriches each annotation with:

- a readable visibility level
- explicit box dimensions
- explicit object coordinates
- ego vehicle position at the corresponding sample

These derived columns will be used in the feature engineering and exploratory analysis sections.

In [14]:
# Create a working copy of the annotation table
preprocessing_df: pd.DataFrame = sample_annotation_df.copy()

## Visibility level

Each annotation includes a `visibility_token`, which can be mapped to a more interpretable visibility level.

In [ ]:
# TODO: should be used later
# Add visibility level

visibility_df: pd.DataFrame = nusc_tables["visibility"]

token_to_visibility: Dict[str, str] = (
    visibility_df.set_index("token")["level"].to_dict()
)

preprocessing_df["visibility_level"] = preprocessing_df["visibility_token"].map(
    token_to_visibility
)

preprocessing_df[["visibility_token", "visibility_level", "category_name"]].head()

## Box dimensions

The `size` field stores the 3D bounding box dimensions in meters as:

- width
- length
- height

These values are split into separate columns for easier analysis.

In [ ]:
# Split box dimensions into separate columns

size_cols: List[str] = ["size_w", "size_l", "size_h"]

for idx, col in enumerate(size_cols):
    preprocessing_df[col] = preprocessing_df["size"].apply(lambda x: x[idx])

preprocessing_df[["size", "size_w", "size_l", "size_h"]].head()

## Translation

The `translation` field stores the 3D box center in meters as `(x, y, z)`.  
For later analysis, this field is split into three separate coordinate columns.

In [ ]:
# Split object translation into x, y, z coordinates
preprocessing_df[["x", "y", "z"]] = pd.DataFrame(
    preprocessing_df["translation"].tolist(),
    index=preprocessing_df.index,
)

preprocessing_df[["translation", "x", "y", "z"]].head()

## Ego vehicle position

To analyze object position relative to the ego vehicle, the ego pose is recovered through the following chain:

`sample_token -> LIDAR_TOP sample_data -> ego_pose_token -> ego position`

In [ ]:
# Map each sample to the corresponding top LiDAR token

sample_df: pd.DataFrame = nusc_tables["sample"]
sample_data_df: pd.DataFrame = nusc_tables["sample_data"]
ego_pose_df: pd.DataFrame = nusc_tables["ego_pose"]

sample_to_lidar_token: pd.Series = sample_df.set_index("token")["data"].apply(
    lambda sensor_dict: sensor_dict.get("LIDAR_TOP") if isinstance(sensor_dict, dict) else None
)

# Map LiDAR token to ego pose token
lidar_token_to_ego_pose_token: pd.Series = sample_data_df.set_index("token")["ego_pose_token"]

# Map ego pose token to ego vehicle translation
ego_pose_token_to_translation: pd.Series = ego_pose_df.set_index("token")["translation"]

# Add intermediate columns
preprocessing_df["lidar_token"] = preprocessing_df["sample_token"].map(sample_to_lidar_token)
preprocessing_df["ego_pose_token"] = preprocessing_df["lidar_token"].map(lidar_token_to_ego_pose_token)
preprocessing_df["ego_position"] = preprocessing_df["ego_pose_token"].map(ego_pose_token_to_translation)

# Split ego position into coordinates
preprocessing_df[["ego_x", "ego_y", "ego_z"]] = pd.DataFrame(
    preprocessing_df["ego_position"].tolist(),
    index=preprocessing_df.index,
)

preprocessing_df[["ego_position", "ego_x", "ego_y", "ego_z"]].head()

In [ ]:
# Preview the processed dataframe
preprocessing_df.head()

# Feature engineering

This section derives analysis-ready features from the preprocessed annotation table. The engineered features are designed to characterize:

- object distance from the ego vehicle
- object geometry
- LiDAR support
- radar support
- camera visibility and projected box size

These features will later support both the exploratory analysis and the design of fusion-oriented modeling strategies.

In [ ]:
# Initialize feature dataframe
features_df: pd.DataFrame = preprocessing_df.copy()
features_df.head()

## Distance-based features

To study detection difficulty as a function of range, 2D and 3D distances from the object center to the ego vehicle are computed. A binned distance variable is also added for grouped analysis.

In [ ]:
# Distance to ego vehicle
features_df["distance_ego_2d"] = np.sqrt(
    (features_df["x"] - features_df["ego_x"]) ** 2 +
    (features_df["y"] - features_df["ego_y"]) ** 2
)

features_df["distance_ego_3d"] = np.sqrt(
    (features_df["x"] - features_df["ego_x"]) ** 2 +
    (features_df["y"] - features_df["ego_y"]) ** 2 +
    (features_df["z"] - features_df["ego_z"]) ** 2
)

distance_bins: List[float] = [0, 10, 20, 30, 40, 50, 70, 100, float("inf")]
distance_labels: List[str] = [
    "0-10", "10-20", "20-30", "30-40",
    "40-50", "50-70", "70-100", "100+"
]

features_df["distance_bin"] = pd.cut(
    features_df["distance_ego_2d"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True,
    right=False,
)

features_df[
    [
        "category_name",
        "x", "y", "z",
        "ego_x", "ego_y", "ego_z",
        "distance_ego_2d", "distance_ego_3d", "distance_bin",
    ]
].head()

## Geometric features

Additional geometric features are derived from the 3D bounding box dimensions, including box volume and aspect ratio.

In [ ]:
# Box volume
features_df["volume"] = (
    features_df["size_w"] *
    features_df["size_l"] *
    features_df["size_h"]
)

features_df[["size_w", "size_l", "size_h", "volume"]].head()

In [ ]:
# Aspect ratio

n_zero: int = int((features_df["size_w"] == 0).sum())

if n_zero > 0:
    print(
        f"Warning: {n_zero} rows have size_w = 0. "
        "aspect_ratio_l_w is set to NaN for these rows."
    )

features_df["aspect_ratio_l_w"] = np.where(
    features_df["size_w"] == 0,
    np.nan,
    features_df["size_l"] / features_df["size_w"],
)

features_df[["size_l", "size_w", "aspect_ratio_l_w"]].head()

## LiDAR support features

LiDAR support is approximated using the number of LiDAR points falling inside each annotated 3D bounding box.

In [ ]:
# LiDAR support flags

features_df["no_lidar_support"] = features_df["num_lidar_pts"] == 0
features_df["low_lidar_support"] = features_df["num_lidar_pts"].between(1, 4)
features_df["usable_lidar_support"] = features_df["num_lidar_pts"] >= 5

features_df[
    [
        "category_name",
        "num_lidar_pts",
        "no_lidar_support",
        "low_lidar_support",
        "usable_lidar_support",
    ]
].head()

## Radar support features

Radar support is approximated using the number of radar points falling inside each annotated 3D bounding box.

In [ ]:
# Radar support flags

features_df["no_radar_support"] = features_df["num_radar_pts"] == 0
features_df["low_radar_support"] = features_df["num_radar_pts"].between(1, 2)
features_df["usable_radar_support"] = features_df["num_radar_pts"] >= 3

features_df[
    [
        "category_name",
        "num_radar_pts",
        "no_radar_support",
        "low_radar_support",
        "usable_radar_support",
    ]
].head()

In [26]:
# TODO: Is it used later
# Category-level radar support grouping

pct_no_radar_by_cat: pd.Series = (
    features_df.groupby("category_name")["num_radar_pts"]
    .apply(lambda s: (s == 0).mean())
)

def radar_support_group(pct_no_radar: float) -> str:
    if pct_no_radar < 0.40:
        return "radar_well_supported"
    if pct_no_radar <= 0.70:
        return "radar_partial"
    return "radar_weak"

features_df["radar_support_group_cat"] = (
    features_df["category_name"]
    .map(pct_no_radar_by_cat)
    .apply(radar_support_group)
)

## Camera visibility features

Camera-related features are derived by checking, for each annotation, which camera views contain the object and how large the projected 3D box appears in the image plane.

The main outputs are:

- number of visible camera views
- names of visible cameras
- whether the object is visible in at least one camera
- maximum projected box area
- mean projected box area
- best camera view

In [27]:
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import BoxVisibility, view_points

In [ ]:
# Adapted from nuscenes.utils.geometry_utils.box_in_image:
# project box corners into the image and keep only corners with positive depth.

def box_area_px(
    box: Box,
    intrinsic: np.ndarray,
    img_w: int,
    img_h: int,
) -> float:
    """
    Compute the projected 2D box area in image pixels.

    The box must already be expressed in camera coordinates.
    """
    corners_3d: np.ndarray = box.corners()
    corners_2d: np.ndarray = view_points(corners_3d, intrinsic, normalize=True)[:2, :]

    in_front: np.ndarray = corners_3d[2, :] > 0.1

    if not np.any(in_front):
        return 0.0

    x: np.ndarray = np.clip(corners_2d[0, in_front], 0, img_w)
    y: np.ndarray = np.clip(corners_2d[1, in_front], 0, img_h)

    width: float = max(0.0, float(x.max() - x.min()))
    height: float = max(0.0, float(y.max() - y.min()))

    return float(width * height)

def get_camera_features(nusc: NuScenes, ann_token: str) -> Dict[str, Any]:
    """
    Compute camera visibility features for one annotation token.
    """
    ann: Dict[str, Any] = nusc.get("sample_annotation", ann_token)
    sample: Dict[str, Any] = nusc.get("sample", ann["sample_token"])

    visible_cam_names: List[str] = []
    visible_cam_areas: List[float] = []

    for sensor_name, sd_token in sample["data"].items():
        if not sensor_name.startswith("CAM"):
            continue

        sd: Dict[str, Any] = nusc.get("sample_data", sd_token)
        img_w: int = sd["width"]
        img_h: int = sd["height"]

        _, boxes, camera_intrinsic = nusc.get_sample_data(
            sd_token,
            box_vis_level=BoxVisibility.ANY,
            selected_anntokens=[ann_token],
        )

        if not boxes:
            continue

        area: float = box_area_px(
            box=boxes[0],
            intrinsic=camera_intrinsic,
            img_w=img_w,
            img_h=img_h,
        )

        visible_cam_names.append(sensor_name)
        visible_cam_areas.append(area)

    if visible_cam_areas:
        best_idx: int = int(np.argmax(visible_cam_areas))
        max_box_area_px: float = float(np.max(visible_cam_areas))
        mean_box_area_px: float = float(np.mean(visible_cam_areas))
        best_camera_name: Optional[str] = visible_cam_names[best_idx]
    else:
        max_box_area_px = 0.0
        mean_box_area_px = 0.0
        best_camera_name = None

    return {
        "num_visible_cams": len(visible_cam_names),
        "visible_cam_names": visible_cam_names,
        "has_any_camera_view": len(visible_cam_names) > 0,
        "max_box_area_px": max_box_area_px,
        "mean_box_area_px": mean_box_area_px,
        "best_camera_name": best_camera_name,
    }
    
# Test on one annotation
test_ann_token: str = features_df.iloc[0]["annotation_token"]
test_result: Dict[str, Any] = get_camera_features(nusc, test_ann_token)

print("Test annotation token:", test_ann_token)
pprint(test_result)

In [29]:
from pathlib import Path
import pandas as pd


def add_camera_feature_columns(df: pd.DataFrame, nusc: NuScenes) -> pd.DataFrame:
    """
    Add camera-derived visibility features to a dataframe of annotations.
    """
    camera_cols: List[str] = [
        "num_visible_cams",
        "visible_cam_names",
        "has_any_camera_view",
        "max_box_area_px",
        "mean_box_area_px",
        "best_camera_name",
    ]

    df = df.drop(columns=camera_cols, errors="ignore").copy()

    unique_tokens: np.ndarray = df["annotation_token"].unique()

    feature_map: Dict[str, Dict[str, Any]] = {
        ann_token: get_camera_features(nusc, ann_token)
        for ann_token in tqdm(unique_tokens, desc="Computing camera features")
    }

    camera_features_df: pd.DataFrame = (
        df["annotation_token"]
        .map(feature_map)
        .apply(pd.Series)
    )
    camera_features_df.index = df.index

    return pd.concat([df, camera_features_df], axis=1)

In [ ]:
# Save the computed full-dataset features_df to avoid the repetiion of the long computation in future runs
CACHE_PATH = Path(
    "/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/cache/features_df_camera_full.pkl"
)

FORCE_RECOMPUTE = False

if DATASET == "full":
    if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
        print("Loading saved features_df...")
        features_df = pd.read_pickle(CACHE_PATH)
    else:
        print("Computing features_df and saving it...")
        features_df = add_camera_feature_columns(features_df, nusc)

        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        features_df.to_pickle(CACHE_PATH)
else:
    print("Mini dataset: computing features_df directly...")
    features_df = add_camera_feature_columns(features_df, nusc)

In [31]:
# TODO: save features?
# camera_features_df.to_parquet("../data/camera_features.parquet", index=False)
# camera_features_df = pd.read_parquet("../data/camera_features.parquet")

In [ ]:
features_df[
    [
        "annotation_token",
        "num_visible_cams",
        "visible_cam_names",
        "has_any_camera_view",
        "max_box_area_px",
        "mean_box_area_px",
        "best_camera_name",
    ]
].head()

In [ ]:
# Camera view quality flags based on projected box area
q25: float = float(features_df["max_box_area_px"].quantile(0.25))
q50: float = float(features_df["max_box_area_px"].quantile(0.50))

features_df["no_camera_support"] = features_df["max_box_area_px"] <= q25
features_df["low_camera_support"] = features_df["max_box_area_px"].between(
    q25,
    q50,
    inclusive="right",
)

features_df["usable_camera_support"] = features_df["max_box_area_px"] > q50

print(features_df["no_camera_support"].value_counts())
print(features_df["low_camera_support"].value_counts())
print(features_df["usable_camera_support"].value_counts())

In [ ]:
# Final preview of engineered features
features_df.head()

# Exploratory Data Analysis

This section examines the statistical structure of the annotation dataset, the intrinsic difficulty of different object categories, and the reliability and complementarity of the available sensing modalities.

## Sanity checks

Before analyzing difficulty and sensor support, it is useful to inspect the overall annotation distribution. In particular, I will check how object annotations are distributed across distance and category, and identify whether some categories are too rare for stable statistical analysis.

In [35]:
# Working copy for EDA
raw_eda_df: pd.DataFrame = features_df.copy()

In [ ]:
# Distance distribution
plt.figure(figsize=(8, 5))
raw_eda_df["distance_ego_2d"].hist(bins=50)
plt.title("Object distance from ego vehicle")
plt.xlabel("Distance (m)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Annotation count by category before filtering
raw_class_counts: pd.Series = (
    raw_eda_df["category_name"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(15, 8))
raw_class_counts.plot(kind="bar")
plt.title("Annotation count by category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Clean the dataframe for statistical analysis

To avoid unstable results driven by extremely rare categories, categories with fewer than 30 annotations are removed from the statistical analysis.

In [ ]:
# Keep only categories with at least 30 annotations
min_count: int = 30

raw_class_counts: pd.Series = raw_eda_df["category_name"].value_counts()

kept_categories = raw_class_counts[raw_class_counts >= min_count].index
removed_categories = raw_class_counts[raw_class_counts < min_count].index

category_filter_summary: pd.DataFrame = pd.DataFrame({
    "category_name": raw_class_counts.index,
    "count": raw_class_counts.values,
    "kept_for_analysis": raw_class_counts.values >= min_count,
})

# Check the condition
category_filter_summary

In [39]:
# Working copy for EDA
eda_df: pd.DataFrame = raw_eda_df[
    raw_eda_df["category_name"].isin(kept_categories)
].copy()

In [ ]:
# Annotation count by category after filtering
eda_class_counts: pd.Series = (
    eda_df["category_name"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(15, 8))
eda_class_counts.plot(kind="bar")
plt.title("Annotation count by category after filtering")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Top categories versus the remaining long tail
top_n: int = 7

top_counts: pd.Series = eda_class_counts.iloc[:top_n]
other_count: int = int(eda_class_counts.iloc[top_n:].sum())

pie_counts: pd.Series = top_counts.copy()
pie_counts["Other"] = other_count

other_categories: pd.Series = eda_class_counts.iloc[top_n:]

plt.figure(figsize=(8, 8))
plt.pie(
    pie_counts.values,
    labels=pie_counts.index,
    autopct="%1.1f%%",
    startangle=90,
)
plt.title(f"Top {top_n} categories versus other categories")
plt.tight_layout()
plt.show()

print("Categories included in 'Other':")
print(list(other_categories.index))

### Key observations

- The distance distribution is strongly skewed toward short and medium ranges, with most annotations below about 50 m.
- The number of objects drops quickly as distance increases, and only a small tail remains at large distances.
- The class distribution is highly imbalanced, with `vehicle.car` clearly dominating the dataset.
- `human.pedestrian.adult` and `movable_object.barrier` are also frequent, but still much less common than cars.
- A small set of categories accounts for most annotations, while many classes appear only rarely.

### Implications for machine learning or fusion

- A baseline model will likely learn common and close-range objects better than rare or distant ones.
- Global metrics alone may hide weak performance on tail classes or far objects, so evaluation should also be done by class and by distance range.
- The strong class imbalance means the model may focus on dominant categories unless this is handled during training or analysis.
- Fusion may be most useful in harder cases, such as distant objects or underrepresented classes where single-sensor evidence may be weaker.

## Intrinsic object difficulty

This subsection examines whether some object categories are inherently more difficult to detect because of their physical properties and visibility, independently of sensor support.

The analysis focuses on:

- object dimensions by category
- object volume by category
- object aspect ratio by category
- object visibility by category
- compound hard-category patterns

### Object by box dimension

In [ ]:
def plot_median_size_by_category(df: pd.DataFrame) -> None:
    """
    Plot median object width, length, and height by category.
    """
    median_sizes: pd.DataFrame = (
        df.groupby("category_name")[["size_w", "size_l", "size_h"]]
        .median()
        .sort_values("size_l")
    )

    median_sizes.plot(kind="bar", figsize=(12, 8))
    plt.title("Median object dimensions by category")
    plt.xlabel("Category")
    plt.ylabel("Meters")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()
    
plot_median_size_by_category(eda_df)

### Object volume by category

Object volume helps identify categories that are physically small and may therefore be harder to detect reliably.

In [ ]:
def plot_box_by_category(
    df: pd.DataFrame,
    y_col: str,
    title: str,
    ylabel: str,
    x_col: str = "category_name",
    figsize: Tuple[int, int] = (10, 10),
    order_by_median: bool = True,
    log_scale: bool = False,
    rotate_xticks: int = 90,
) -> None:
    """
    Plot the distribution of a numeric feature by category.
    """
    plot_df: pd.DataFrame = df.dropna(subset=[x_col, y_col]).copy()

    order = None
    if order_by_median:
        order = (
            plot_df.groupby(x_col)[y_col]
            .median()
            .sort_values()
            .index
        )

    plt.figure(figsize=figsize)
    sns.boxplot(
        data=plot_df,
        x=x_col,
        y=y_col,
        order=order,
    )

    plt.title(title)
    plt.xlabel("Category")
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotate_xticks, ha="right")

    if log_scale:
        plt.yscale("log")

    plt.tight_layout()
    plt.show()
    
plot_box_by_category(
    df=eda_df,
    y_col="volume",
    title="Object volume by category",
    ylabel="Volume (m³)",
    log_scale=True,
)

In [ ]:
def get_low_classes_by_median(
    df: pd.DataFrame,
    value_col: str,
    threshold: float,
    category_col: str = "category_name",
) -> pd.DataFrame:
    """
    Return categories whose median value is below a specified threshold.
    """
    summary_df: pd.DataFrame = (
        df.dropna(subset=[category_col, value_col])
        .groupby(category_col)[value_col]
        .median()
        .sort_values()
        .reset_index()
    )

    summary_df.columns = [category_col, f"median_{value_col}"]

    return summary_df.loc[summary_df[f"median_{value_col}"] < threshold].copy()

low_volume_df: pd.DataFrame = get_low_classes_by_median(
    df=eda_df,
    value_col="volume",
    threshold=1.0,
)

low_volume_df

### Object aspect ratio by category

Aspect ratio provides a simple proxy for object shape. Categories with more elongated or less compact shapes may be harder to localize consistently.

In [ ]:
plot_box_by_category(
    df=eda_df,
    y_col="aspect_ratio_l_w",
    title="Object aspect ratio by category",
    ylabel="Length-to-width ratio",
    log_scale=True,
)

low_aspect_ratio_df: pd.DataFrame = get_low_classes_by_median(
    df=eda_df,
    value_col="aspect_ratio_l_w",
    threshold=1.0,
)

low_aspect_ratio_df

### Object visibility by category

Visibility is analyzed by category to identify object classes that are more frequently observed under low-visibility conditions.

In [ ]:
vis_by_class: pd.DataFrame = (
    pd.crosstab(
        index=eda_df["category_name"],
        columns=eda_df["visibility_level"],
        normalize="index",
    )
    .sort_values("v0-40", ascending=False)
)

vis_by_class

In [ ]:
vis_by_class.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Visibility distribution by category")
plt.xlabel("Category")
plt.ylabel("Proportion")
plt.legend(title="Visibility", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
low_visibility_df: pd.DataFrame = vis_by_class.loc[vis_by_class["v0-40"] > 0.25].copy()
low_visibility_df

### Hard categories summary

The previous indicators are combined to identify categories that appear difficult across multiple intrinsic dimensions.

In [49]:
low_visibility_set = set(low_visibility_df.index)
low_aspect_ratio_set = set(low_aspect_ratio_df["category_name"])
low_volume_set = set(low_volume_df["category_name"])

In [ ]:
category_intrinsic_flags: pd.DataFrame = pd.DataFrame({
    "category_name": sorted(eda_df["category_name"].unique()),
})

category_intrinsic_flags["low_visibility"] = (
    category_intrinsic_flags["category_name"].isin(low_visibility_set)
)
category_intrinsic_flags["low_aspect_ratio"] = (
    category_intrinsic_flags["category_name"].isin(low_aspect_ratio_set)
)
category_intrinsic_flags["low_volume"] = (
    category_intrinsic_flags["category_name"].isin(low_volume_set)
)

category_intrinsic_flags["n_intrinsic_difficulty_flags"] = (
    category_intrinsic_flags[["low_visibility", "low_aspect_ratio", "low_volume"]]
    .sum(axis=1)
)

category_intrinsic_flags["hard_in_at_least_two"] = (
    category_intrinsic_flags["n_intrinsic_difficulty_flags"] >= 2
)

category_intrinsic_flags["hard_in_all_three"] = (
    category_intrinsic_flags["n_intrinsic_difficulty_flags"] == 3
)

category_intrinsic_flags = category_intrinsic_flags.sort_values(
    ["n_intrinsic_difficulty_flags", "category_name"],
    ascending=[False, True],
).reset_index(drop=True)

category_intrinsic_flags

In [ ]:
hard_in_all_three = set(
    category_intrinsic_flags.loc[
        category_intrinsic_flags["hard_in_all_three"],
        "category_name",
    ]
)

print("Hard in all three:")
pprint(hard_in_all_three)

hard_in_at_least_two = set(
    category_intrinsic_flags.loc[
        category_intrinsic_flags["hard_in_at_least_two"],
        "category_name",
    ]
)

print("\nHard in at least two:")
pprint(hard_in_at_least_two)

In [ ]:
print("Low visibility + low aspect ratio:")
print(low_visibility_set & low_aspect_ratio_set)

print("\nLow visibility + low volume:")
print(low_visibility_set & low_volume_set)

print("\nLow aspect ratio + low volume:")
print(low_aspect_ratio_set & low_volume_set)

In [ ]:
hard_rows: int = int(
    eda_df[eda_df["category_name"].isin(hard_in_at_least_two)].shape[0]
)
all_rows: int = int(eda_df.shape[0])

print("Intrinsic hard categories:")
print(sorted(hard_in_at_least_two))

print(f"\nPercentage of rows from hard categories: {100 * hard_rows / all_rows:.2f}%")

### Key observations

- The pedestrian-related categories tend to have smaller median dimensions and lower object volumes than most vehicle classes, which makes them intrinsically harder to detect.
- `human.pedestrian.construction_worker` is the only category marked as hard in all three intrinsic signals: low visibility, low aspect ratio, and low volume.
- Several categories are hard in at least two signals, mainly pedestrian-related classes plus `movable_object.debris` and `movable_object.pushable_pullable`.
- Low visibility and low volume overlap for `human.pedestrian.adult`, `human.pedestrian.child`, `human.pedestrian.construction_worker`, `human.pedestrian.stroller`, and `movable_object.pushable_pullable`.
- Low aspect ratio and low volume overlap for `human.pedestrian.construction_worker`, `human.pedestrian.police_officer`, and `movable_object.debris`.
- These intrinsically hard categories account for about 21.34% of all rows, so they are not a small corner case in the dataset.

### Implications for machine learning or fusion

- Detection difficulty is not only a sensor issue; some categories are hard because of their intrinsic shape, size, and visibility properties.
- Pedestrian-related classes should be treated as an important difficult group when evaluating model performance and failure modes.
- Since hard categories make up a meaningful share of the data, performance on them should be tracked separately rather than hidden inside global metrics.
- Fusion may be especially useful for categories that are small and less visible, where a single modality may provide weak evidence.
- The baseline should include category-wise analysis to check whether errors are concentrated in these intrinsically hard classes.

## Sensor reliability by regime

This section characterizes the reliability of each sensing modality across distance, visibility, and object category. The goal is to identify where each sensor is informative, weak, or absent before moving to fusion complementarity.

### Helper functions

In [54]:
def summarize_flag_by_group(
    df: pd.DataFrame,
    group_col: str,
    flag_col: str,
    sort_index: bool = True,
) -> pd.DataFrame:
    """
    Return the within-group proportion of False/True values for a boolean flag.
    """
    table: pd.DataFrame = pd.crosstab(
        index=df[group_col],
        columns=df[flag_col],
        normalize="index",
    )

    if sort_index:
        table = table.sort_index()

    return table 

def plot_stacked_proportion_table(
    table: pd.DataFrame,
    title: str,
    xlabel: str,
    ylabel: str = "Proportion",
    figsize: Tuple[int, int] = (8, 4),
    rot: int = 0,
    ha: str = "center",
) -> None:
    """
    Plot a stacked bar chart from a summary table of proportions.
    Rows are groups, columns are mutually exclusive categories.
    """
    ax = table.plot(
        kind="bar",
        stacked=True,
        figsize=figsize,
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=rot, ha=ha)

    ax.legend(
        title="Support level",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
    )

    plt.tight_layout()
    plt.show()
    
def plot_bar_from_table(
    table: pd.DataFrame,
    true_col: bool = True,
    title: str = "",
    ylabel: str = "",
    xlabel: str = "",
    figsize: Tuple[int, int] = (8, 4),
    rot: int = 0,
    ha: str = "center",
) -> None:
    """
    Plot the proportion associated with the True column from a boolean summary table.
    """
    plot_series: pd.Series = table[true_col] if true_col in table.columns else pd.Series(0, index=table.index)

    ax = plot_series.plot.bar(figsize=figsize, title=title, rot=rot)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=rot, ha=ha)
    plt.tight_layout()
    plt.show()
    
def make_support_summary_table(
    df: pd.DataFrame,
    group_col: str,
    flag_cols: List[str],
) -> pd.DataFrame:
    """
    Return the percentage of annotations in each boolean support group by regime.
    """
    summary_df: pd.DataFrame = (
        df.groupby(group_col)[flag_cols]
        .mean()
        .mul(100)
        .round(1)
    )
    return summary_df

### Camera reliability

In [55]:
camera_flags: Dict[str, str] = {
    "no_camera_support": "No camera support",
    "low_camera_support": "Low camera support",
    "usable_camera_support": "Usable camera support",
}

#### Camera reliability by distance

In [ ]:
for flag, label in camera_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "distance_bin", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin",
    )

#### Camera reliability by visibility

In [ ]:
for flag, label in camera_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "visibility_level", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level",
    )

#### Camera reliability by category

In [ ]:
camera_flags.keys()

In [ ]:
camera_by_category: pd.DataFrame = (
    eda_df.groupby("category_name")[
        ["no_camera_support", "low_camera_support", "usable_camera_support"]
    ]
    .mean()
    .rename(columns={
        "no_camera_support": camera_flags["no_camera_support"],
        "low_camera_support": camera_flags["low_camera_support"],
        "usable_camera_support": camera_flags["usable_camera_support"],
    })
    .sort_values(camera_flags["usable_camera_support"], ascending=False)
)

plot_stacked_proportion_table(
    table=camera_by_category,
    title="Camera reliability by category",
    xlabel="Category",
    figsize=(12, 8),
    rot=90,
    ha="right",
)

#### Camera summary tables

In [ ]:
camera_support_summary_by_distance: pd.DataFrame = make_support_summary_table(
    eda_df,
    "distance_bin",
    ["no_camera_support", "low_camera_support", "usable_camera_support"],
)

camera_support_summary_by_distance

In [ ]:
camera_support_summary_by_distance: pd.DataFrame = make_support_summary_table(
    eda_df,
    "distance_bin",
    ["no_camera_support", "low_camera_support", "usable_camera_support"],
)

camera_support_summary_by_distance

#### Key observations

- Camera support drops steadily with distance. Good camera view falls from 96.8% at 0–10 m to 2.2% at 100+ m, while weak camera view rises from 1.4% to 85.5%.
- Limited camera view is most common at mid range. It increases up to about 36.3% in the 50–70 m bin, then decreases again at the longest distances.
- Visibility also affects camera reliability, but the effect is weaker than distance. Good camera view increases from about 42% in the lowest visibility group to about 54% in the highest one.
- Weak camera view is highest in the lowest visibility group, but the gap between visibility levels is much smaller than the gap across distance bins.
- Camera reliability varies by category. Trailers, buses, construction vehicles, and wheelchairs have relatively high proportions of good camera support, while traffic cones, children, adults, and animals have much higher weak camera support.
- Pedestrian-related classes and small objects appear less camera-reliable than large vehicle classes in this analysis.

#### Implications for machine learning or fusion

- Distance is a major driver of camera weakness, so fusion should be especially important for far-range objects.
- Since visibility has a smaller effect than distance here, distance-based analysis may be more useful than visibility alone when defining hard camera regimes.
- Categories with weak camera support, such as traffic cones and several pedestrian classes, are good candidates for stronger multimodal support.
- Camera-only performance is likely to degrade on small objects and distant objects, so these cases should be checked separately in evaluation.
- A fusion strategy that adapts to object regime, especially distance and category, is likely to be more useful than treating all cases the same.

### LiDAR reliability

In [62]:
lidar_flags: Dict[str, str] = {
    "no_lidar_support": "No LiDAR support",
    "low_lidar_support": "Low LiDAR support",
    "usable_lidar_support": "Usable LiDAR support",
}

#### LiDAR reliability by distance

In [ ]:
for flag, label in lidar_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "distance_bin", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin",
    )

#### LiDAR reliability by visibility

In [ ]:
for flag, label in lidar_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "visibility_level", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level",
    )

#### LiDAR reliability by category

In [ ]:
lidar_by_distance: pd.DataFrame = (
    eda_df.groupby("category_name")[
        ["no_lidar_support", "low_lidar_support", "usable_lidar_support"]
    ]
    .mean()
    .rename(columns={
        "no_lidar_support": lidar_flags["no_lidar_support"],
        "low_lidar_support": lidar_flags["low_lidar_support"],
        "usable_lidar_support": lidar_flags["usable_lidar_support"],
    })
    .sort_values(lidar_flags["usable_lidar_support"], ascending=False)
)

plot_stacked_proportion_table(
    table=lidar_by_distance,
    title="LiDAR reliability by category",
    xlabel="category",
    figsize=(12, 8),
    rot=90,
    ha="right",
)

#### LiDAR summary tables

In [ ]:
lidar_support_summary_by_distance: pd.DataFrame = make_support_summary_table(
    eda_df,
    "distance_bin",
    ["no_lidar_support", "low_lidar_support", "usable_lidar_support"],
)

lidar_support_summary_by_distance

In [ ]:
lidar_support_summary_by_visibility: pd.DataFrame = make_support_summary_table(
    eda_df,
    "visibility_level",
    ["no_lidar_support", "low_lidar_support", "usable_lidar_support"],
)

lidar_support_summary_by_visibility

#### Key observations

- LiDAR support drops strongly with distance. Usable LiDAR support falls from 96.9% at 0–10 m to 2.9% at 100+ m, while no LiDAR support rises from 0.9% to 77.2%.
- Low LiDAR support is most common at mid range. It increases up to about 48.9% in the 40–50 m bin, then decreases at longer distances as no-support cases become dominant.
- Visibility has a clear effect on LiDAR reliability. Usable LiDAR support rises from 21.1% in the lowest visibility group to 63.3% in the highest one.
- No LiDAR support is much higher in poor visibility, dropping from 38.6% at v0–40 to 10.2% at v80–100.
- LiDAR reliability also varies by category. Wheelchairs, buses, trailers, and bicycle racks have relatively high usable support, while traffic cones, children, strollers, animals, and adult pedestrians have higher no-support or low-support proportions.
- Compared with the visibility plots, distance appears to have the stronger effect, but visibility also matters clearly for LiDAR support.

#### Implications for machine learning or fusion

- Distance is a major source of LiDAR weakness, so fusion should be especially important for far-range objects.
- Since LiDAR support also improves with visibility, hard regimes should consider both distance and visibility rather than distance alone.
- Small and pedestrian-related categories appear less LiDAR-reliable, so they are good candidates for stronger multimodal support.
- LiDAR-only performance is likely to weaken for distant objects and some small object classes, so these cases should be evaluated separately.
- A fusion strategy that adapts to object regime, especially distance, visibility, and category, is likely to be more useful than a single uniform strategy.

### Radar reliability

In [68]:
radar_flags: Dict[str, str] = {
    "no_radar_support": "No radar support",
    "low_radar_support": "Low radar support",
    "usable_radar_support": "Usable radar support",
}

#### Radar reliability by distance

In [ ]:
for flag, label in radar_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "distance_bin", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by distance",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Distance bin",
    )

#### Radar reliability by visibility

In [ ]:
for flag, label in radar_flags.items():
    table: pd.DataFrame = summarize_flag_by_group(eda_df, "visibility_level", flag)

    plot_bar_from_table(
        table=table,
        title=f"{label} by visibility",
        ylabel=f"Proportion of {label.lower()}",
        xlabel="Visibility level",
    )

#### Radar reliability by category

In [ ]:
radar_by_distance: pd.DataFrame = (
    eda_df.groupby("category_name")[
        ["no_radar_support", "low_radar_support", "usable_radar_support"]
    ]
    .mean()
    .rename(columns={
        "no_radar_support": radar_flags["no_radar_support"],
        "low_radar_support": radar_flags["low_radar_support"],
        "usable_radar_support": radar_flags["usable_radar_support"],
    })
    .sort_values(radar_flags["usable_radar_support"], ascending=False)
)

plot_stacked_proportion_table(
    table=radar_by_distance,
    title="usable_radar_support",
    xlabel="category",
    figsize=(12, 8),
    rot=90,
    ha="right",
)


#### Radar summary tables

In [ ]:
radar_support_summary_by_distance: pd.DataFrame = make_support_summary_table(
    eda_df,
    "distance_bin",
    ["no_radar_support", "low_radar_support", "usable_radar_support"],
)

radar_support_summary_by_distance

In [ ]:
radar_support_summary_by_visibility: pd.DataFrame = make_support_summary_table(
    eda_df,
    "visibility_level",
    ["no_radar_support", "low_radar_support", "usable_radar_support"],
)

radar_support_summary_by_visibility

#### Key observations

- Radar support is weak across almost all distance bins. No radar support stays high, around 61% to 73%, and usable radar support remains low in every range.
- Distance still matters for radar, but the effect is smaller than for camera or LiDAR. Usable radar support drops from about 16% to 18% at close range to about 6% at 100+ m.
- Visibility also affects radar reliability. Usable radar support rises from about 5% in the lowest visibility group to about 19% in the highest one.
- Even in the best visibility group, no radar support is still the largest share, so radar remains unreliable for many objects.
- Radar reliability varies strongly by category. Large vehicle classes such as buses, trailers, trucks, police vehicles, and ambulances have much higher usable radar support than pedestrian-related classes and small movable objects.
- Pedestrian classes, traffic cones, debris, barriers, and strollers show very high no-radar support, often with only a very small usable radar share.

#### Implications for machine learning or fusion

- Radar is unlikely to be a strong standalone source for many objects, so its main value may come from complementing other sensors in selected cases.
- Radar seems more promising for large vehicle categories than for pedestrians or small objects, so category-aware fusion is likely to be more useful than a uniform fusion strategy.
- Since usable radar support stays low across distance and visibility regimes, adding radar may not help equally in all situations and should be justified by regime-specific analysis.
- Evaluation should check whether radar improves hard vehicle cases rather than expecting broad gains across all classes.
- For pedestrian-heavy or small-object regimes, camera and LiDAR are likely to remain more important than radar based on these plots.

##  Sensor comparison

Compare how camera and LiDAR support change across distance
bins on the same plot.

### by distance


In [ ]:
from typing import List, Tuple
import pandas as pd
import matplotlib.pyplot as plt

comparison_triplets: List[Tuple[str, str, str]] = [
    ("usable_camera_support", "usable_lidar_support", "usable_radar_support"),
    ("low_camera_support", "low_lidar_support", "low_radar_support"),
    ("no_camera_support", "no_lidar_support", "no_radar_support"),
]

group_col: str = "distance_bin"   # or "visibility_level"

for camera_flag, lidar_flag, radar_flag in comparison_triplets:
    camera_label: str = camera_flags[camera_flag]
    lidar_label: str = lidar_flags[lidar_flag]
    radar_label: str = radar_flags[radar_flag]

    camera_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, camera_flag
    )
    lidar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, lidar_flag
    )
    radar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, radar_flag
    )

    comparison_table: pd.DataFrame = pd.DataFrame({
        camera_label: camera_table[True],
        lidar_label: lidar_table[True],
        radar_label: radar_table[True],
    })

    comparison_table.plot(
        kind="bar",
        figsize=(8, 4),
        title=f"{camera_label} vs {lidar_label} vs {radar_label} by {group_col.replace('_', ' ')}",
    )
    plt.xlabel(group_col.replace("_", " ").title())
    plt.ylabel("Proportion")
    plt.ylim(0, 1)
    plt.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.show()

### by visibility

In [ ]:
group_col: str = "visibility_level"   # or "distance_bin"

for camera_flag, lidar_flag, radar_flag in comparison_triplets:
    camera_label: str = camera_flags[camera_flag]
    lidar_label: str = lidar_flags[lidar_flag]
    radar_label: str = radar_flags[radar_flag]

    camera_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, camera_flag
    )
    lidar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, lidar_flag
    )
    radar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, radar_flag
    )

    comparison_table: pd.DataFrame = pd.DataFrame({
        camera_label: camera_table[True],
        lidar_label: lidar_table[True],
        radar_label: radar_table[True],
    })

    comparison_table.plot(
        kind="bar",
        figsize=(8, 4),
        title=(
            f"{camera_label} vs {lidar_label} vs {radar_label} "
            f"by {group_col.replace('_', ' ')}"
        ),
    )
    plt.xlabel(group_col.replace("_", " ").title())
    plt.ylabel("Proportion")
    plt.ylim(0, 1)
    plt.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.show()

### by category

In [ ]:
from typing import List, Tuple
import pandas as pd
import matplotlib.pyplot as plt

comparison_triplets: List[Tuple[str, str, str]] = [
    ("usable_camera_support", "usable_lidar_support", "usable_radar_support"),
    ("low_camera_support", "low_lidar_support", "low_radar_support"),
    ("no_camera_support", "no_lidar_support", "no_radar_support"),
]

group_col: str = "category_name"

for camera_flag, lidar_flag, radar_flag in comparison_triplets:
    camera_label: str = camera_flags[camera_flag]
    lidar_label: str = lidar_flags[lidar_flag]
    radar_label: str = radar_flags[radar_flag]

    camera_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, camera_flag
    )
    lidar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, lidar_flag
    )
    radar_table: pd.DataFrame = summarize_flag_by_group(
        eda_df, group_col, radar_flag
    )

    comparison_table: pd.DataFrame = pd.DataFrame({
        camera_label: camera_table[True],
        lidar_label: lidar_table[True],
        radar_label: radar_table[True],
    })

    comparison_table = comparison_table.sort_values(
        by=lidar_label,
        ascending=False,
    )

    comparison_table.plot(
        kind="bar",
        figsize=(12, 5),
        title=f"{camera_label} vs {lidar_label} vs {radar_label} by category",
    )
    plt.xlabel("Category")
    plt.ylabel("Proportion")
    plt.ylim(0, 1)
    plt.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=90, ha="right")
    plt.show()

### Key observations

- Camera and LiDAR are much stronger than radar in the usable-support plots, both by distance and by visibility.
- Camera and LiDAR usable support both drop quickly with distance, while radar usable support stays low across all distance bins.
- LiDAR usable support becomes higher than camera usable support at higher visibility levels, while camera is slightly stronger in the lowest visibility group.
- Radar no-support is consistently high across distance, visibility, and many categories, which shows that radar is unreliable for a large share of objects.
- By category, large vehicle classes tend to have higher usable support than pedestrian-related classes and small objects, especially for radar.
- Pedestrian-related classes, traffic cones, debris, barriers, and other small objects often show higher no-support or low-support levels, especially for radar and often also for camera or LiDAR.

### Implications for machine learning or fusion

- Camera and LiDAR should be the main focus of the first fusion baseline, because they provide the strongest and most complementary support across regimes.
- Radar is unlikely to help equally across all cases, so its use should be justified by specific regimes or categories rather than treated as a core input from the start.
- Distance-aware evaluation is important, because both camera and LiDAR weaken clearly with range even when they are much stronger than radar overall.
- Category-aware evaluation is also important, since pedestrian classes and small objects appear harder and less reliably supported than large vehicle classes.
- A practical strategy is to start with a LiDAR-only baseline, then test camera–LiDAR fusion, and treat radar as a later extension if it shows value in selected cases.

## Cross-modal complementarity

This section examines whether one sensing modality remains informative when another becomes weak or unavailable. The goal is to identify regimes where fusion is likely to provide real value beyond unimodal detection.

Rather than enumerating all possible combinations, I focus on a small set of interpretable regimes:
- camera rescues weak range sensing
- range sensing rescues weak camera
- all modalities are strong
- all modalities are weak

These regimes directly capture when fusion is beneficial.

In [77]:
# ------------------------------------------------------------
# One strong modality, two weak modalities
# ------------------------------------------------------------

eda_df["camera_strong_with_lidar_support"] = (
    eda_df["usable_camera_support"]
    & ~eda_df["no_lidar_support"]
)

eda_df["camera_not_good_lidar_low"] = (
    ~eda_df["usable_camera_support"]
    & eda_df["low_lidar_support"]
)



# eda_df["radar_only_strong"] = (
#     ~eda_df["good_camera_view"]
#     & ~eda_df["usable_lidar_support"]
#     & eda_df["usable_radar_support"]
# )

### Helper functions

In [78]:
def summarize_flags_by_group(
    df: pd.DataFrame,
    group_col: str,
    flag_cols: List[str]
) -> pd.DataFrame:
    """Return mean rate of boolean flags by group."""
    return (
        df.groupby(group_col)[flag_cols]
        .mean()
        .sort_index()
    )

def plot_grouped_flag_rates(
    table: pd.DataFrame,
    title: str,
    xlabel: str,
    ylabel: str = "Proportion",
    figsize: tuple = (10, 5),
    rot: int = 0
) -> None:
    """Plot grouped bar chart from grouped flag-rate table."""
    ax = table.plot(kind="bar", figsize=figsize)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=rot)
    plt.legend(
        title="",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )
    plt.tight_layout()
    plt.show()

def plot_flag_by_category(
    df: pd.DataFrame,
    flag_col: str,
    title: str,
    ylabel: str = "Proportion",
    xlabel: str = "Category",
    rot: int = 90,
    ascending: bool = False
) -> pd.DataFrame:
    """Plot mean per-category rate for one boolean flag."""
    table = (
        summarize_flags_by_group(
            df=df,
            group_col="category_name",
            flag_cols=[flag_col]
        )
        .sort_values(by=flag_col, ascending=ascending)
    )

    ax = table[flag_col].plot.bar(figsize=(12, 5), rot=rot)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=rot, ha="right")
    plt.tight_layout()
    plt.show()

    return table

### Global prevalence of complementarity regimes

In [ ]:
complementarity_flags = [
    "camera_strong_with_lidar_support",
    "camera_not_good_lidar_low",
]

complementarity_label_map = {
    "camera_strong_with_lidar_support": "Camera strong with LiDAR support",
    "camera_not_good_lidar_low": "Camera not good and LiDAR low",
}

complementarity_summary = (
    eda_df[complementarity_flags]
    .mean()
    .mul(100)
    .rename(index=complementarity_label_map)
    .sort_values(ascending=False)
    .round(2)
)

print("Camera–LiDAR regime prevalence (% of rows):")
print(complementarity_summary)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    x=complementarity_summary.values,
    y=complementarity_summary.index
)
plt.title("Prevalence of complementarity regimes")
plt.xlabel("Percentage of rows")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Complementarity by distance

In [ ]:
complementarity_by_distance = (
    eda_df.groupby("distance_bin")[complementarity_flags]
    .mean()
    .rename(columns=complementarity_label_map)
)

plot_grouped_flag_rates(
    table=complementarity_by_distance,
    title="Complementarity regimes by distance",
    xlabel="Distance bin",
    ylabel="Proportion",
    rot=90,
    # ha="right",
)

### Complementarity by visibility

In [ ]:
complementarity_by_visibility = (
    eda_df.groupby("visibility_level")[complementarity_flags]
    .mean()
    .rename(columns=complementarity_label_map)
)

plot_grouped_flag_rates(
    table=complementarity_by_visibility,
    title="Complementarity regimes by visibility",
    xlabel="Visibility level",
    ylabel="Proportion",
)

### Complementarity by category

In [ ]:
# ------------------------------------------------------------
# Camera–LiDAR regimes by category as one stacked bar plot
# ------------------------------------------------------------

cam_lidar_by_category = (
    eda_df.groupby("category_name")[
        [
            "camera_strong_with_lidar_support",
            "camera_not_good_lidar_low",
        ]
    ]
    .mean()
)

# Add remaining cases so the stacked bars sum to 1
cam_lidar_by_category["other_regimes"] = (
    1
    - cam_lidar_by_category[
        [
            "camera_strong_with_lidar_support",
            "camera_not_good_lidar_low",
        ]
    ].sum(axis=1)
)

# Rename columns for readability
cam_lidar_by_category = cam_lidar_by_category.rename(
    columns={
        "camera_strong_with_lidar_support": "Camera strong with LiDAR support",
        "camera_not_good_lidar_low": "Camera not good and LiDAR low",
        "other_regimes": "Other regimes",
    }
)

# Sort categories by one regime of interest
cam_lidar_by_category = cam_lidar_by_category.sort_values(
    by="Camera strong with LiDAR support",
    ascending=False
)

plot_stacked_proportion_table(
    table=cam_lidar_by_category,
    title="Camera–LiDAR regimes by category",
    xlabel="Category",
    ylabel="Proportion",
    figsize=(12, 6),
    rot=90,
    ha="right",
)

### Compact summary tables

In [ ]:
print("\nComplementarity regimes by distance:")
display(complementarity_by_distance.round(3))

print("\nComplementarity regimes by visibility:")
display(complementarity_by_visibility.round(3))

### Key observations

- The dominant camera–LiDAR regime is **camera strong with LiDAR support** (~45% of rows), while **camera not good and LiDAR low** covers about 25%.
- The balance changes strongly with distance: the strong regime drops from 96.0% at 0–10 m to 1.4% at 100+, while the weak regime peaks around 39–40% at 40–70 m.
- Around 30–40 m, the two regimes are nearly equal, suggesting a transition zone between easier and harder camera–LiDAR conditions.
- Better visibility increases the strong regime from 30.6% to 52.9%, while the weak regime stays more stable at about 23–28%.
- Large vehicle classes are more often in the strong regime, while pedestrian-related classes, traffic cones, and some small movable objects show more of the weak regime.
- Across categories, the strong regime generally decreases as the weak regime increases, and together these two regimes cover most cases in most categories.

### Implications for machine learning or fusion

- Distance is a key factor for camera–LiDAR complementarity, so fusion should pay special attention to mid- and far-range objects where the strong regime becomes much less common.
- The 30–70 m range looks especially important, because this is where weak camera–LiDAR conditions become common and may need more careful fusion handling.
- Since visibility improves the strong regime, visibility can help define easier versus harder fusion settings, but distance appears to be the stronger driver here.
- Category-aware fusion is likely to help, because pedestrian classes and small objects seem to face weaker camera–LiDAR support than large vehicles.
- Evaluation should separate easy regimes from weak camera–LiDAR regimes, so average results do not hide failures in the harder cases.

## Machine learning

### Machine Learning Research Question

The main research question of this project is whether multimodal fusion can improve 3D object detection on nuScenes compared with a strong single-modality baseline, especially in the regimes that appear difficult in the exploratory analysis.

More specifically, the analysis suggests three linked questions. First, can fusion improve detection for objects at mid and far range, where both camera and LiDAR support drop strongly? Second, can fusion help intrinsically difficult categories, especially pedestrian-related classes and small objects, which are harder because of size, visibility, and shape? Third, can fusion improve performance in weak camera–LiDAR regimes, rather than only improving already easy cases?

These questions focus the machine learning part of the project on robustness in hard conditions, not only on overall average performance.

### Planned Models and Rationale

The first model should be a LiDAR-only baseline. LiDAR is still the strongest single modality for 3D structure, and it provides a practical and standard reference point for later comparison. This baseline is important because it shows what can already be achieved without fusion.

The second model should be a camera–LiDAR fusion baseline. The exploratory analysis suggests that camera and LiDAR are often complementary, but that this complementarity changes with distance, visibility, and category. A camera–LiDAR model is therefore the most relevant first fusion setup.

Radar should not be the main focus of the first baseline. The plots show that radar support is weak for many categories and remains limited across most regimes. Radar may still be useful later, especially for some vehicle classes, but it is less promising as the first fusion path.

This leads to a simple staged plan: start with a LiDAR-only baseline, then add a camera–LiDAR fusion model, and use radar only as a later extension if time and results justify it.

### Detailed Machine Learning Strategy

The practical strategy is to build a stable baseline first, then test whether fusion helps in the difficult regimes identified by the EDA. Training should begin with a standard LiDAR-only detector in MMDetection3D on the full nuScenes train/val split. Once this baseline is working and reproducible, a camera–LiDAR fusion model should be trained under the same evaluation setup.

The evaluation should go beyond a single overall score. Since the dataset is strongly imbalanced and most objects are at short range, global metrics alone may hide important weaknesses. Results should therefore be broken down by distance range, by category, and by difficult groups such as pedestrian-related classes, small objects, and intrinsically hard categories. It is also useful to compare easier camera–LiDAR regimes against weak camera–LiDAR regimes, since the EDA shows a clear transition around 30–70 m.

The modeling choices should follow the main findings from the analysis. Distance should be treated as a major difficulty factor. Category-wise analysis is also necessary because large vehicle classes appear much easier than pedestrians, traffic cones, debris, and other small objects. Visibility should be included in error analysis, even if distance appears to be the stronger driver for camera and LiDAR reliability.

Class imbalance should be handled carefully. Since cars and common pedestrian classes dominate the dataset, the baseline may become biased toward frequent classes. For this reason, the model should be monitored with per-class results and not only aggregate metrics. The first goal is not to solve the full imbalance problem, but to measure clearly where the model succeeds and where it fails.

Overall, the machine learning workflow should be: build a reproducible LiDAR baseline, add a camera–LiDAR fusion baseline, compare them under the same setup, and check whether fusion gives real gains in the hard regimes identified by the EDA.

### Key findings and conclusions

The exploratory analysis gives a clear direction for the modeling stage. Camera and LiDAR both become much weaker with distance, and several pedestrian-related and small-object categories are intrinsically hard because of size, visibility, and shape. These are the cases where fusion is most likely to matter.

The analysis also suggests that radar should not be the first fusion priority. Its support is weak for many objects and seems more useful for selected vehicle cases than for the dataset as a whole. As a result, the most practical first strategy is camera–LiDAR fusion, not full three-modality fusion.

A simple and strong project plan is therefore to start with a LiDAR-only baseline, then test a camera–LiDAR fusion model, and evaluate both carefully by distance, category, and hard regimes. This makes the project both realistic and well aligned with the evidence from the EDA.

In summary, the goal is not only to build a fusion model, but to test whether fusion improves detection where single sensors are weak. This gives the machine learning section a clear motivation, a practical baseline path, and an evaluation strategy that matches the structure of the dataset.